In [ ]:
# ============================================================
#   Single Grayscale Image Encryption using Cross-Coupled PWLCM
# ============================================================



import hashlib
import math
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files

# ─────────────────────────────────────────────
# STEP 1: Upload your image
# ─────────────────────────────────────────────
print("Please upload a grayscale or color image (it will be converted to grayscale):")
uploaded = files.upload()
image_filename = list(uploaded.keys())[0]

# Load and convert to grayscale
img = Image.open(image_filename).convert('L')
original = np.array(img, dtype=np.uint8)
print(f"Image loaded: {image_filename}  |  Size: {original.shape[1]}x{original.shape[0]} px")


# ─────────────────────────────────────────────
# PWLCM: Piece-wise Linear Chaotic Map
# ─────────────────────────────────────────────
def pwlcm(s, a):
    if 0 <= s < a:
        return s / a
    elif a <= s < 0.5:
        return (s - a) / (0.5 - a)
    else:
        return 1.0 - s


# ─────────────────────────────────────────────
# KEY GENERATION via SHA-256
# ─────────────────────────────────────────────
def generate_keys(image, xm, xx1, ym, yx1):
    hash_hex = hashlib.sha256(image.tobytes()).hexdigest()   # 64 hex chars
    hd = [int(hash_hex[i], 16) for i in range(64)]

    def _key(original, digits):
        s = sum(digits)
        return original - math.floor(s / 1e15) - math.ceil(s / 1e15) * 0.01

    eps = 1e-10
    x1 = np.clip(abs(_key(xm,  hd[0:16])),  eps, 1 - eps)
    ux = np.clip(abs(_key(xx1, hd[16:32])), eps, 0.5 - eps)
    y1 = np.clip(abs(_key(ym,  hd[32:48])), eps, 1 - eps)
    yx = np.clip(abs(_key(yx1, hd[48:64])), eps, 0.5 - eps)

    return x1, ux, y1, yx, hash_hex


# ─────────────────────────────────────────────
# CROSS-COUPLED SEQUENCE GENERATION
# ─────────────────────────────────────────────
def cross_coupled_sequences(x1, ux, y1, yx, Mx):
    x_seq = np.zeros(Mx)
    y_seq = np.zeros(Mx)
    x_seq[0], y_seq[0] = x1, y1
    x_seq[1] = pwlcm(x1, ux)
    y_seq[1] = pwlcm(y1, yx)

    for i in range(1, Mx - 1):
        yi = y_seq[i] if y_seq[i] <= 0.5 else 1.0 - y_seq[i]
        x_seq[i + 1] = pwlcm(yi, ux)
        xi = x_seq[i + 1] if x_seq[i + 1] <= 0.5 else 1.0 - x_seq[i + 1]
        y_seq[i + 1] = pwlcm(xi, yx)

    return x_seq, y_seq


# ─────────────────────────────────────────────
# ENCRYPTION
# ─────────────────────────────────────────────
def encrypt(image,
            xm=0.25686446985353, xx1=0.35488659076447,
            ym=0.26457834689785, yx1=0.36789543267894):

    M, N = image.shape
    Mx   = max(M, N)

    # Keys
    x1, ux, y1, yx, hash_hex = generate_keys(image, xm, xx1, ym, yx1)
    print(f"\nGenerated keys:")
    print(f"  x(1) = {x1:.15f}   ux = {ux:.15f}")
    print(f"  y(1) = {y1:.15f}   yx = {yx:.15f}")
    print(f"  SHA-256 hash: {hash_hex}")

    # Cross-coupled sequences
    x_seq, y_seq = cross_coupled_sequences(x1, ux, y1, yx, Mx)

    # Permutation indices (sort ascending)
    row_index = np.argsort(x_seq[(Mx - M):])
    col_index = np.argsort(y_seq[(Mx - N):])

    # Row-Column Permutation (shuffle)
    permuted = image[row_index, :][:, col_index]

    # Key sequences for diffusion (mod to uint8)
    x_raw = (np.round(x_seq[(Mx - M):] * 1e6) % 256).astype(np.uint8)
    y_raw = (np.round(y_seq[(Mx - N):] * 1e6) % 256).astype(np.uint8)
    x_key = np.tile(x_raw, math.ceil(N / M))[:N]   # length N
    y_key = np.tile(y_raw, math.ceil(M / N))[:M]   # length M

    # Row Diffusion (XOR chain across rows)
    row_diff = np.zeros_like(permuted)
    prev = x_key.copy()
    for i in range(M):
        row_diff[i] = permuted[i] ^ prev
        prev = row_diff[i]

    # Column Diffusion (XOR chain across columns)
    col_diff = np.zeros_like(row_diff)
    prev = y_key.copy()
    for j in range(N):
        col_diff[:, j] = row_diff[:, j] ^ prev
        prev = col_diff[:, j]

    return col_diff, hash_hex


# ─────────────────────────────────────────────
# STEP 2: Run Encryption
# ─────────────────────────────────────────────
cipher, sha_hash = encrypt(original)
print("\nEncryption complete!")


# ─────────────────────────────────────────────
# STEP 3: Display original and encrypted images
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("Original Image", fontsize=14)
axes[0].axis('off')

axes[1].imshow(cipher, cmap='gray', vmin=0, vmax=255)
axes[1].set_title("Encrypted Image", fontsize=14)
axes[1].axis('off')

plt.suptitle("Cross-Coupled PWLCM Image Encryption", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("encryption_result.png", dpi=150, bbox_inches='tight')
plt.show()
print("Side-by-side comparison saved as: encryption_result.png")


# ─────────────────────────────────────────────
# STEP 5: Save and download encrypted image
# ─────────────────────────────────────────────
enc_filename = "encrypted_" + image_filename.rsplit('.', 1)[0] + ".png"
Image.fromarray(cipher).save(enc_filename)
print(f"\nEncrypted image saved as: {enc_filename}")

files.download(enc_filename)
files.download("encryption_result.png")